# 智能体

Agent = Model + Harness

Harness：智能体运行环境。一个 harness 就是围绕那个循环的一切：模型、它的提示、它的工具，以及任何塑造其行为的中间件。

create_agent 是一个高度可配置的 Harness。

## 核心组件

- Model
- Tools
- System prompt
- Structured output


### Model
create_agent的 model 参数，可以是一个```provider:model_name```字符串，也可以是一个```BaseChatModel```对象。

In [ ]:
from langchain.chat_models import init_chat_model
import os


llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1",
)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model="google_genai:gemini-3.5-flash", tools=[])

llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1",
)
agent = create_agent(model=llm, tools=[])

### Tools

tools 可以传入任何 Python 可调用对象、LangChain 工具或工具字典。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

tools=[search]
agent = create_agent(model=llm, tools=tools)

### System prompt

系统提示词参数可以接受一个字符串或系统消息。

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)


### Structured output

结构化输出，用于将模型生成的文本结果转换为结构化数据。

In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent


class Answer(BaseModel):
    summary: str
    confidence: float


agent = create_agent(model=llm, tools=tools, response_format=Answer)
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
result["structured_response"]  # Answer(summary=..., confidence=...)

## Invocation

- state：状态，保存着当前对话的会话信息，如历史对话记录。
- config：配置，保存着当前对话的配置信息，如configurable thread_id等。
- context：上下文，保存着当前对话的上下文信息，如user_id，session_id等。
- checkpointer：检查点，保存着当前对话的检查点信息，必须有checkpointer，才会为agent提供底层存储。

In [ ]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
from dataclasses import dataclass

config = {"configurable": {"thread_id": str(uuid7())}}

@dataclass
class Context:
    user_id: str

agent = create_agent(
    model=llm, 
    tools=tools, 
    checkpointer=InMemorySaver(),
    context_schema=Context,
)

context=Context(user_id="user-123")

result = agent.invoke(
    {"messages": [{"role": "user", "content": "我是艾尔文，你叫什么?"}]},
    config=config,
    context=context
)

for message in result["messages"]:
    message.pretty_print()

print('-' * 180)

# A follow-up turn on the same conversation: reuse the same thread_id to keep history
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我是谁?"}]},
    config=config,
    context=context
)
for message in result["messages"]:
    message.pretty_print()

## Streaming

使用流式传输实时显示中间消息和工具活动。

In [ ]:
from langchain.messages import AIMessage, HumanMessage


stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]},
    config=config,
    context=context,
    version="v3",
)
for snapshot in stream.values:
    # Each snapshot contains the full state at that point
    latest_message = snapshot["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"User: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

## Configure the harness

create_agent的中间件机制，可以自定义扩展agent。langchain提供了一些预置的中间件，如：

- `FilesystemMiddleware`: 文件系统中间件，读写文件、执行脚本。
- `SummarizationMiddleware`: 语义总结中间件，对输入进行语义总结，例如：`SummarizationMiddleware(model=model, backend=backend)`
- `MemoryMiddleware`: 内存中间件，存储和检索数据。例如：`MemoryMiddleware(backend=backend, sources=["./AGENTS.md"])`
- `SkillsMiddleware`: 技能中间件，执行技能。例如：`SkillsMiddleware(backend=backend, sources=["./skills/"])`
- `TodoListMiddleware`: 计划中间件，管理待办事项。例如：`TodoListMiddleware()`
- `SubAgentMiddleware`: 子代理中间件，创建子代理。例如：`SubAgentMiddleware(...)`
- `ModelRetryMiddleware`: 模型重试中间件，对模型调用进行重试。例如：`ModelRetryMiddleware(max_retries=3)`
- `ToolRetryMiddleware`: 工具重试中间件，对工具调用进行重试。例如：`ToolRetryMiddleware(max_retries=3)`
- `PIIMiddleware`: PII中间件，对输入进行PII过滤。例如：`PIIMiddleware("email")`
- `HumanInTheLoopMiddleware`: 人机交互中间件，对输入进行人机交互。例如：`HumanInTheLoopMiddleware(interrupt_on={"write_file": True})`


### 执行环境

执行环境为代理提供了一个工作区：它可以调用的工具、一个用于跨回合读写文件的文件系统，以及运行脚本或命令行指令的代码执行功能。

In [ ]:
from langchain.agents import create_agent
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware

agent = create_agent(
    model=llm,
    tools=[search],
    middleware=[FilesystemMiddleware(backend=StateBackend())],
)

### 上下文管理

每次模型调用都有一个固定的上下文窗口。当一个代理运行时，这个窗口会随着历史记录、工具结果和中间步骤的累积而填满。通过上下文管理，代理可以控制这个窗口的大小，并确保它不会超出限制。总结会在窗口溢出之前压缩历史；内存会在启动时加载持久化指令，这样知识可以跨会话保留；技能则按需展示领域知识，而不是一开始就加载所有内容。

In [ ]:
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware, MemoryMiddleware, SkillsMiddleware, SummarizationMiddleware

backend = StateBackend()

agent = create_agent(
    model=llm,
    tools=[search],
    middleware=[
        FilesystemMiddleware(backend=backend),
        SummarizationMiddleware(model=llm, backend=backend),
        MemoryMiddleware(backend=backend, sources=["./AGENTS.md"]),
        SkillsMiddleware(backend=backend, sources=["./skills/"]),
    ],
)

### 计划和分工

复杂的任务通常超出了单个上下文窗口能处理的范围。委派让主代理可以把工作分解成小块，交给在各自独立上下文中运行的子代理处理，而自己则专注于协调而不是执行。工作可以并行进行；主代理的上下文保持干净。

In [ ]:
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware
from deepagents.middleware.subagents import SubAgentMiddleware
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for a query and return a short summary."""
    return f"Search results for: {query}"


backend = StateBackend()

agent = create_agent(
    model=llm,
    tools=[search],
    middleware=[
        FilesystemMiddleware(backend=backend),
        TodoListMiddleware(),
        SubAgentMiddleware(
            backend=backend,
            subagents=[
                {
                    "name": "researcher",
                    "description": "Searches and returns a structured summary.",
                    "system_prompt": "Use the search tool to research the question and summarize key points.",
                    "tools": [search],
                    "model": "anthropic:claude-sonnet-4-6",
                    "middleware": [],
                }
            ],
        ),
    ],
)

### 代理命名

可以选择为代理使用一个标识符。当在多代理系统中将代理作为子图嵌入时，这尤其有用。

In [9]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=tools, name="research_assistant")

### 容错

生产环境中的代理会遇到在开发环境中很少出现的故障：速率限制、模型超时、临时的 API 错误。容错中间件在基础设施层面处理这些问题，这样你的工具和业务逻辑就不需要在每次调用周围都写 try/catch。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for a query and return a short summary."""
    return f"Search results for: {query}"


agent = create_agent(
    model=llm,
    tools=[search],
    middleware=[
        ModelRetryMiddleware(max_retries=3),
        ToolRetryMiddleware(max_retries=2),
    ],
)

### 护栏
有些策略不能仅靠提示来执行——它们需要被确定性地强制执行，无论模型做什么。护栏会在数据通过代理循环时进行拦截，在工具结果到达模型上下文之前应用合规规则或内容政策。

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for a query and return a short summary."""
    return f"Search results for: {query}"


agent = create_agent(
    model=llm,
    tools=[search],
    middleware=[PIIMiddleware("email")],
)

### 控制

完全自主并不总是合适的。控制功能让你可以在人类需要做出决策的关键节点介入——在破坏性写操作、昂贵的 API 调用或任何需要判断的地方——而无需重构你的智能体。智能体会暂停等待；人类批准、编辑或拒绝；然后执行继续。

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for a query and return a short summary."""
    return f"Search results for: {query}"


agent = create_agent(
    model=llm,
    tools=[search],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"write_file": True})],
)